# Усиленная модель прогноза ухода клиента

В этом ноутбуке мы усиливаем первую модель и переходим к более серьёзной постановке:

- используем несколько исторических дат-срезов;
- расширяем набор признаков;
- проверяем модели на временном разделении;
- выбираем порог риска под задачу удержания.

## Почему этот вариант сильнее

Первая модель была полезным стартом, но у неё было ограничение: она училась только на одном временном срезе.

Здесь мы делаем более правильную схему:

- собираем несколько срезов во времени;
- обучаемся на прошлых периодах;
- проверяемся на более позднем периоде;
- сравниваем несколько алгоритмов;
- выбираем модель не только по одной метрике, а по сочетанию качества и пользы для удержания.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 300)
sns.set_theme(style="whitegrid", context="talk")

BASE_DIR = Path("..")
DATA_PATH = BASE_DIR / "data.csv"
EVENTS_PATH = BASE_DIR / "events.csv"


In [2]:
def parse_dt(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, utc=True, errors="coerce", format="mixed")


def safe_mode(series: pd.Series):
    mode = series.mode(dropna=True)
    if not mode.empty:
        return mode.iat[0]
    non_null = series.dropna()
    return non_null.iat[0] if len(non_null) else np.nan


def top_share_precision(y_true: pd.Series, y_score: np.ndarray, top_share: float = 0.2) -> float:
    n_top = max(1, int(len(y_score) * top_share))
    idx = np.argsort(-y_score)[:n_top]
    return y_true.iloc[idx].mean()


def evaluate_scores(y_true: pd.Series, scores: np.ndarray, threshold: float = 0.5) -> dict:
    pred = (scores >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, scores),
        "pr_auc": average_precision_score(y_true, scores),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "top_10pct_precision": top_share_precision(y_true.reset_index(drop=True), scores, top_share=0.1),
        "top_20pct_precision": top_share_precision(y_true.reset_index(drop=True), scores, top_share=0.2),
    }


## 1. Загружаем и подготавливаем исходные данные

In [3]:
data_cols = [
    "order_id", "order_item_id", "user_id", "status", "gender", "created_at", "returned_at",
    "shipped_at", "delivered_at", "sale_price", "age", "state", "city", "traffic_source",
    "category", "department", "brand", "product_id", "warehouse_name", "is_loyal"
]
event_cols = [
    "id", "user_id", "session_id", "sequence_number", "created_at", "traffic_source",
    "browser", "uri", "event_type"
]

data = pd.read_csv(DATA_PATH, usecols=data_cols)
events = pd.read_csv(EVENTS_PATH, usecols=event_cols)

for col in ["created_at", "returned_at", "shipped_at", "delivered_at"]:
    data[col] = parse_dt(data[col])
events["created_at"] = parse_dt(events["created_at"])

data = data.drop_duplicates(subset=["order_item_id"]).copy()
events = events.drop_duplicates(subset=["id"]).copy()

data["city"] = data["city"].fillna("unknown")
data["brand"] = data["brand"].fillna("unknown")
data["sale_price"] = data["sale_price"].clip(lower=0)

data["ship_delay_days"] = (data["shipped_at"] - data["created_at"]).dt.total_seconds() / 86400
data["delivery_days"] = (data["delivered_at"] - data["shipped_at"]).dt.total_seconds() / 86400
data["return_days"] = (data["returned_at"] - data["delivered_at"]).dt.total_seconds() / 86400
data["is_returned"] = data["returned_at"].notna().astype(int)
data["is_cancelled"] = data["status"].eq("Cancelled").astype(int)
data["is_delivered"] = data["delivered_at"].notna().astype(int)

orders = (
    data.groupby("order_id", as_index=False)
    .agg(
        user_id=("user_id", "first"),
        created_at=("created_at", "min"),
        order_value=("sale_price", "sum"),
        items_count=("order_item_id", "nunique"),
        status=("status", "first"),
        is_loyal=("is_loyal", "max"),
    )
    .dropna(subset=["user_id", "created_at"])
)
orders["created_at_naive"] = orders["created_at"].dt.tz_localize(None)

events = events.dropna(subset=["user_id", "created_at"]).copy()
events["user_id"] = events["user_id"].astype(int)
events["created_at_naive"] = events["created_at"].dt.tz_localize(None)
events["is_product_event"] = events["event_type"].eq("product").astype(int)
events["is_cart_event"] = events["event_type"].eq("cart").astype(int)
events["is_purchase_event"] = events["event_type"].eq("purchase").astype(int)
events["is_department_event"] = events["event_type"].eq("department").astype(int)

orders.shape, events.shape


((125085, 8), (1308418, 14))

## 2. Формируем несколько временных срезов

Каждый срез устроен так:

- смотрим историю клиента за последние `365` дней до даты-среза;
- строим признаки только по этой истории;
- если в следующие `180` дней покупок нет, ставим признак ухода.

Используем такие даты-срезы:

- `2024-06-01`
- `2024-09-01`
- `2024-12-01`
- `2025-03-01`
- `2025-06-01`


In [4]:
LOOKBACK_DAYS = 365
CHURN_HORIZON_DAYS = 180
CUTOFF_DATES = pd.to_datetime([
    "2024-06-01",
    "2024-09-01",
    "2024-12-01",
    "2025-03-01",
    "2025-06-01",
])


def build_slice(cutoff_date: pd.Timestamp) -> pd.DataFrame:
    lookback_start = cutoff_date - pd.Timedelta(days=LOOKBACK_DAYS)
    horizon_end = cutoff_date + pd.Timedelta(days=CHURN_HORIZON_DAYS)

    history_orders = orders.loc[orders["created_at_naive"] < cutoff_date].copy()
    window_orders = history_orders.loc[
        (history_orders["created_at_naive"] >= lookback_start) &
        (history_orders["created_at_naive"] < cutoff_date)
    ].copy()

    user_history = (
        history_orders.groupby("user_id", as_index=False)
        .agg(
            total_orders_before_cutoff=("order_id", "nunique"),
            loyal_flag=("is_loyal", "max"),
            first_order_ever=("created_at_naive", "min"),
            last_order_ever=("created_at_naive", "max"),
        )
    )

    eligible = user_history.loc[
        (user_history["total_orders_before_cutoff"] >= 2) | (user_history["loyal_flag"] == True)
    ].copy()

    future_buyers = set(
        orders.loc[
            (orders["created_at_naive"] >= cutoff_date) &
            (orders["created_at_naive"] < horizon_end),
            "user_id"
        ].unique()
    )

    base = eligible[["user_id", "total_orders_before_cutoff", "loyal_flag", "first_order_ever", "last_order_ever"]].copy()
    base["cutoff_date"] = cutoff_date
    base["customer_age_days"] = (cutoff_date - base["first_order_ever"]).dt.days
    base["days_since_last_order_ever"] = (cutoff_date - base["last_order_ever"]).dt.days
    base["churn_target"] = (~base["user_id"].isin(future_buyers)).astype(int)

    order_features = (
        window_orders.groupby("user_id", as_index=False)
        .agg(
            orders_last_365d=("order_id", "nunique"),
            revenue_last_365d=("order_value", "sum"),
            avg_order_value_last_365d=("order_value", "mean"),
            median_order_value_last_365d=("order_value", "median"),
            max_order_value_last_365d=("order_value", "max"),
            min_order_value_last_365d=("order_value", "min"),
            avg_items_per_order_last_365d=("items_count", "mean"),
            max_items_per_order_last_365d=("items_count", "max"),
            last_order_in_window=("created_at_naive", "max"),
            first_order_in_window=("created_at_naive", "min"),
        )
    )
    order_features["days_since_last_order"] = (cutoff_date - order_features["last_order_in_window"]).dt.days
    order_features["active_days_in_window"] = (order_features["last_order_in_window"] - order_features["first_order_in_window"]).dt.days
    order_features["orders_per_active_day"] = order_features["orders_last_365d"] / order_features["active_days_in_window"].replace(0, np.nan)

    data_window = data.loc[
        (data["created_at"].dt.tz_localize(None) >= lookback_start) &
        (data["created_at"].dt.tz_localize(None) < cutoff_date)
    ].copy()

    product_features = (
        data_window.groupby("user_id", as_index=False)
        .agg(
            order_items_last_365d=("order_item_id", "count"),
            distinct_products_last_365d=("product_id", "nunique"),
            distinct_categories_last_365d=("category", "nunique"),
            distinct_brands_last_365d=("brand", "nunique"),
            returned_items_count=("is_returned", "sum"),
            cancelled_items_count=("is_cancelled", "sum"),
            delivered_items_count=("is_delivered", "sum"),
            return_rate_items=("is_returned", "mean"),
            cancel_rate_items=("is_cancelled", "mean"),
            avg_ship_delay_days=("ship_delay_days", "mean"),
            median_delivery_days=("delivery_days", "median"),
            avg_return_days=("return_days", "mean"),
            avg_sale_price=("sale_price", "mean"),
            median_sale_price=("sale_price", "median"),
            age=("age", "median"),
            gender=("gender", "first"),
            state=("state", lambda s: safe_mode(s)),
            city=("city", lambda s: safe_mode(s)),
            main_order_source=("traffic_source", lambda s: safe_mode(s)),
            main_department=("department", lambda s: safe_mode(s)),
            main_category=("category", lambda s: safe_mode(s)),
            main_brand=("brand", lambda s: safe_mode(s)),
            main_warehouse=("warehouse_name", lambda s: safe_mode(s)),
        )
    )
    product_features["return_to_delivery_ratio"] = product_features["returned_items_count"] / product_features["delivered_items_count"].replace(0, np.nan)
    product_features["distinct_categories_per_order"] = product_features["distinct_categories_last_365d"] / product_features["order_items_last_365d"].replace(0, np.nan)

    events_window = events.loc[
        (events["created_at_naive"] >= lookback_start) &
        (events["created_at_naive"] < cutoff_date)
    ].copy()

    event_features = (
        events_window.groupby("user_id", as_index=False)
        .agg(
            sessions_last_365d=("session_id", "nunique"),
            events_last_365d=("id", "count"),
            product_views_last_365d=("is_product_event", "sum"),
            cart_events_last_365d=("is_cart_event", "sum"),
            purchase_events_last_365d=("is_purchase_event", "sum"),
            department_events_last_365d=("is_department_event", "sum"),
            avg_sequence_last_365d=("sequence_number", "mean"),
            max_sequence_last_365d=("sequence_number", "max"),
            last_event_at=("created_at_naive", "max"),
            main_browser=("browser", lambda s: safe_mode(s)),
            main_event_source=("traffic_source", lambda s: safe_mode(s)),
            main_event_type=("event_type", lambda s: safe_mode(s)),
        )
    )
    event_features["days_since_last_event"] = (cutoff_date - event_features["last_event_at"]).dt.days
    event_features["events_per_session_last_365d"] = event_features["events_last_365d"] / event_features["sessions_last_365d"].replace(0, np.nan)
    event_features["cart_to_view_ratio"] = event_features["cart_events_last_365d"] / event_features["product_views_last_365d"].replace(0, np.nan)
    event_features["purchase_to_cart_ratio"] = event_features["purchase_events_last_365d"] / event_features["cart_events_last_365d"].replace(0, np.nan)

    slice_df = (
        base
        .merge(order_features, on="user_id", how="left")
        .merge(product_features, on="user_id", how="left")
        .merge(event_features, on="user_id", how="left")
    )

    return slice_df


In [5]:
dataset = pd.concat([build_slice(cutoff) for cutoff in CUTOFF_DATES], ignore_index=True)
dataset.shape


(84535, 63)

In [6]:
slice_summary = dataset.groupby("cutoff_date", as_index=False).agg(
    clients=("user_id", "count"),
    churn_share=("churn_target", "mean")
)
slice_summary


,cutoff_date,clients,churn_share
0,2024-06-01,13336,0.829259
1,2024-09-01,14916,0.824082
2,2024-12-01,16711,0.820238
3,2025-03-01,18686,0.817564
4,2025-06-01,20886,0.806138


## 3. Делаем честное временное разделение

Обучаемся на ранних периодах, проверяем на более поздних:

- обучение: `2024-06-01`, `2024-09-01`, `2024-12-01`
- проверка: `2025-03-01`
- финальный тест: `2025-06-01`

Так мы проверяем, умеет ли модель переноситься во времени.

In [7]:
train_cutoffs = pd.to_datetime(["2024-06-01", "2024-09-01", "2024-12-01"])
valid_cutoff = pd.Timestamp("2025-03-01")
test_cutoff = pd.Timestamp("2025-06-01")

drop_cols = [
    "user_id", "first_order_ever", "last_order_ever", "last_order_in_window", "first_order_in_window",
    "last_event_at", "cutoff_date", "churn_target"
]
feature_cols = [col for col in dataset.columns if col not in drop_cols]

train_df = dataset[dataset["cutoff_date"].isin(train_cutoffs)].copy()
valid_df = dataset[dataset["cutoff_date"] == valid_cutoff].copy()
test_df = dataset[dataset["cutoff_date"] == test_cutoff].copy()

X_train = train_df[feature_cols].copy()
y_train = train_df["churn_target"].copy()
X_valid = valid_df[feature_cols].copy()
y_valid = valid_df["churn_target"].copy()
X_test = test_df[feature_cols].copy()
y_test = test_df["churn_target"].copy()

categorical_cols = X_train.select_dtypes(include=["object", "bool"]).columns.tolist()
numeric_cols = [col for col in X_train.columns if col not in categorical_cols]

train_df.shape, valid_df.shape, test_df.shape


((44963, 63), (18686, 63), (20886, 63))

## 4. Сравниваем несколько моделей

Будем сравнивать:

- логистическую регрессию как понятную базовую модель;
- случайный лес;
- `Extra Trees` как более сильную ансамблевую модель;
- `HistGradientBoosting` на числовых признаках.

Последняя модель особенно полезна, если нелинейные зависимости важнее простых линейных правил.

In [8]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

full_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

numeric_only_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]), numeric_cols),
    ],
    remainder="drop",
)

models = {
    "Логистическая регрессия": Pipeline([
        ("preprocessor", full_preprocessor),
        ("model", LogisticRegression(max_iter=3000, class_weight="balanced")),
    ]),
    "Случайный лес": Pipeline([
        ("preprocessor", full_preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=350,
            max_depth=10,
            min_samples_leaf=15,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "Extra Trees": Pipeline([
        ("preprocessor", full_preprocessor),
        ("model", ExtraTreesClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=8,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "HistGradientBoosting": Pipeline([
        ("preprocessor", numeric_only_preprocessor),
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_depth=6,
            max_iter=350,
            min_samples_leaf=40,
            random_state=42,
        )),
    ]),
}


In [9]:
valid_results = []
fitted_models = {}
valid_scores = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[model_name] = model
    scores = model.predict_proba(X_valid)[:, 1]
    valid_scores[model_name] = scores
    metrics = evaluate_scores(y_valid, scores, threshold=0.5)
    metrics["model"] = model_name
    valid_results.append(metrics)

valid_results_df = pd.DataFrame(valid_results).sort_values(["pr_auc", "roc_auc"], ascending=False)
valid_results_df


,roc_auc,pr_auc,balanced_accuracy,precision,recall,top_10pct_precision,top_20pct_precision,model
2,0.800202,0.943377,0.724876,0.932049,0.667998,0.980728,0.974311,Extra Trees
0,0.789866,0.939644,0.710244,0.913271,0.732015,0.974304,0.973241,Логистическая регрессия
1,0.778546,0.935606,0.687742,0.904802,0.710480,0.973769,0.972438,Случайный лес
3,0.724694,0.915164,0.538115,0.829205,0.990574,0.973233,0.941129,HistGradientBoosting


## 5. Подбираем рабочий порог риска на проверочном периоде

Для удержания нам обычно не нужен стандартный порог `0.5`. Но и слишком низкий порог опасен: тогда модель начнёт помечать почти всех клиентов как рискованных.

- не пропускать слишком много реально уходящих клиентов;
- при этом не помечать как рискованных вообще всех подряд;
- учитывать, сколько клиентов бизнес реально может обработать удерживающей кампанией.

Ниже считаем два режима:

- `Сбалансированный режим` — хороший общий компромисс между точностью и полнотой;
- `Режим кампании` — когда мы работаем только с верхними `20%` самых рискованных клиентов.

In [10]:
best_model_name = valid_results_df.iloc[0]["model"]
best_model = fitted_models[best_model_name]
best_valid_scores = valid_scores[best_model_name]

precisions, recalls, thresholds = precision_recall_curve(y_valid, best_valid_scores)
threshold_table = pd.DataFrame(
    {
        "threshold": np.append(thresholds, 1.0),
        "precision": precisions,
        "recall": recalls,
    }
)
threshold_table["f1_score"] = 2 * threshold_table["precision"] * threshold_table["recall"] / (threshold_table["precision"] + threshold_table["recall"])
threshold_table["f1_score"] = threshold_table["f1_score"].fillna(0)

balanced_threshold = threshold_table.sort_values(["f1_score", "precision"], ascending=False).iloc[0]["threshold"]
campaign_threshold = np.quantile(best_valid_scores, 0.8)

pd.DataFrame(
    {
        "mode": ["Сбалансированный", "Кампания top 20%"],
        "threshold": [balanced_threshold, campaign_threshold],
    }
)


,threshold,precision,recall,business_score
396,0.321242,0.829634,0.993258,0.927808
399,0.321550,0.829715,0.993192,0.927801
392,0.320749,0.829507,0.993323,0.927797
395,0.320958,0.829588,0.993258,0.927790
398,0.321358,0.829670,0.993192,0.927783
391,0.320743,0.829462,0.993323,0.927779
411,0.323024,0.830041,0.992931,0.927775
394,0.320932,0.829543,0.993258,0.927772
397,0.321292,0.829624,0.993192,0.927765
407,0.322849,0.829914,0.992996,0.927763


In [11]:
selected_threshold = balanced_threshold
selected_threshold


0.3212415906248958

## 6. Финальная проверка на тестовом периоде

In [12]:
test_scores = best_model.predict_proba(X_test)[:, 1]
test_metrics_default = evaluate_scores(y_test, test_scores, threshold=0.5)
test_metrics_balanced = evaluate_scores(y_test, test_scores, threshold=float(balanced_threshold))
test_metrics_campaign = evaluate_scores(y_test, test_scores, threshold=float(campaign_threshold))

comparison = pd.DataFrame([
    {"setup": "Порог 0.5", **test_metrics_default},
    {"setup": f"Сбалансированный порог {float(balanced_threshold):.3f}", **test_metrics_balanced},
    {"setup": f"Кампания top 20% / порог {float(campaign_threshold):.3f}", **test_metrics_campaign},
])
comparison


,setup,roc_auc,pr_auc,balanced_accuracy,precision,recall,top_10pct_precision,top_20pct_precision
0,Порог 0.5,0.808787,0.939689,0.732160,0.922657,0.712775,0.974617,0.969835
1,Порог 0.321,0.808787,0.939689,0.532157,0.816358,0.995902,0.974617,0.969835


In [13]:
test_pred = (test_scores >= float(selected_threshold)).astype(int)
campaign_pred = (test_scores >= float(campaign_threshold)).astype(int)
print('Лучшая модель:', best_model_name)
print('Сбалансированный порог:', round(float(selected_threshold), 4))
print(classification_report(y_test, test_pred, digits=3))
print('\nРежим кампании top 20%:')
print(classification_report(y_test, campaign_pred, digits=3))


Лучшая модель: Extra Trees
Порог: 0.3212
              precision    recall  f1-score   support

           0      0.801     0.068     0.126      4049
           1      0.816     0.996     0.897     16837

    accuracy                          0.816     20886
   macro avg      0.808     0.532     0.512     20886
weighted avg      0.813     0.816     0.748     20886



## 7. Какие признаки важнее всего

In [ ]:
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=5,
    random_state=42,
)

importance_df = pd.DataFrame(
    {
        "feature": X_test.columns,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    }
).sort_values("importance_mean", ascending=False)
importance_df.head(20)


In [ ]:
top_features = importance_df.head(15).sort_values("importance_mean")

plt.figure(figsize=(12, 9))
plt.barh(top_features["feature"], top_features["importance_mean"], color="#2f6db3")
plt.title(f"Важность признаков: {best_model_name}")
plt.xlabel("Падение ROC-AUC при перемешивании признака")
plt.tight_layout()
plt.show()


## 8. Как переводить результат в бизнес-решения

Если в числе самых важных признаков окажутся такие показатели, как:

- `days_since_last_order`;
- `orders_last_365d`;
- `sessions_last_365d`;
- `events_per_session_last_365d`;
- `return_rate_items`;
- `cancel_rate_items`;
- `median_delivery_days`;
- `purchase_to_cart_ratio`;

это значит, что модель действительно опирается на содержательные сигналы:

- давность и частота покупок;
- цифровую активность клиента;
- проблемность покупательского опыта;
- качество доставки;
- снижение интереса к покупке.

Такие признаки легко использовать не только в модели, но и в стратегии удержания.

Практический вывод по порогу:

- если нужен общий флаг риска, используем `сбалансированный порог`;
- если команда удержания ограничена по ресурсу, лучше брать верхние `10-20%` клиентов по риску, а не пытаться работать со всеми сразу.

## Что делать дальше

После этого ноутбука логично двигаться в две стороны одновременно:

1. Улучшать прогноз ухода:
- добавить ещё временные срезы;
- сделать более точные признаки по категориям и возвратам;
- попробовать калибровку вероятностей.

2. Связать уход с рекомендациями:
- для клиентов с высоким риском показывать более безопасные и подходящие товары;
- исключать категории с повышенным возвратом;
- строить отдельный сценарий для лояльных клиентов.